# AstroCLIP Teaching Lab (Quicklook — Pretrained Checkpoints)

This is a **fast-forward version** of `astroclip_teaching_standalone.ipynb` for
use when there isn't time to train live in front of a class. Instead of
training the image autoencoder, spectrum autoencoder, and CLIP alignment
model from scratch, this notebook **loads pretrained checkpoints** produced
by a previous full run and jumps straight to the results: reconstructions,
the joint embedding space, and the downstream redshift regression.

Run `astroclip_teaching_standalone.ipynb` (or `teaching_scripts/train_*.py`)
at least once first to produce the checkpoints this notebook loads from
`astroclip_demo/*.ckpt`. See `README_astroclip.md` for full setup
instructions — the environment, dataset preparation, and working-directory
requirements are identical to the full notebook.

## 0. Environment Prerequisites

Same environment as the full notebook — see `README_astroclip.md`. Launch
Jupyter from `Y2obs/notebooks/` so the local `astroclip`/`teaching_scripts`
packages and the default dataset/checkpoint paths resolve correctly.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import lightning as L

from datasets import load_dataset, load_from_disk, DatasetDict

import h5py
import umap

from astroclip.data.datamodule import AstroClipCollator

from teaching_scripts.data_utils import (
    load_astroclip_dataset,
    build_image_dataloader,
    build_spectrum_dataloader,
    build_multimodal_dataloader,
)
from teaching_scripts.models import (
    ImageAutoencoder,
    SpectrumAutoencoder,
    SmallCLIPModel,
)

ASTROCLIP_ROOT = Path(os.environ.get('ASTROCLIP_ROOT', './astroclip_demo')).resolve()
ASTROCLIP_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_PATH = ASTROCLIP_ROOT / 'teaching_notebook_subset'
print(f"ASTROCLIP_ROOT: {ASTROCLIP_ROOT}")
print(f"Dataset path:  {DATASET_PATH}")

### Load Subset

Same prepared dataset subset used by the full notebook.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET_PATH} not found. Prepare it first with "
        "teaching_scripts/prepare_dataset.py (see README_astroclip.md)."
    )

ds = load_from_disk(DATASET_PATH)
print(ds)
display({split: len(ds[split]) for split in ds})

### Inspect Sample

Visualise a few image/spectrum pairs to confirm the dataset loaded correctly.

In [ ]:
collator = AstroClipCollator(center_crop=144)

samples = []
for i in range(4):
    item = dict(ds['train'][i])
    item['image'] = np.asarray(item['image'])
    item['spectrum'] = np.asarray(item['spectrum'])
    samples.append(item)

batch = collator(samples)
images = batch['image']
spectra_raw = [np.asarray(s) for s in batch['spectrum']]
print('Image batch shape:', images.shape)
print('Raw spectrum shape example:', spectra_raw[0].shape)

spectra = np.stack([spec.reshape(-1) for spec in spectra_raw])
print('Flattened spectra shape:', spectra.shape)

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for ax, img in zip(axes, images):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.axis('off')
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(spectra[0])
plt.title('Example spectrum')
plt.xlabel('Pixel')
plt.ylabel('Flux')
plt.show()


## 1. Image Autoencoder (loaded from checkpoint)

Loads the pretrained weights instead of training — same architecture,
same 144×144 RGB crops.

In [ ]:
image_ckpt_path = ASTROCLIP_ROOT / 'image_autoencoder_notebook.ckpt'
image_train_loader = build_image_dataloader(ds['train'], batch_size=128, shuffle=True, num_workers=0)
image_val_loader = build_image_dataloader(ds['test'], batch_size=128, shuffle=False, num_workers=0)

if not image_ckpt_path.exists():
    raise FileNotFoundError(
        f"{image_ckpt_path} not found. Run astroclip_teaching_standalone.ipynb "
        "at least once first to produce this checkpoint."
    )

image_model = ImageAutoencoder.load_from_checkpoint(image_ckpt_path)
image_model.eval()
print(f"Loaded pretrained image autoencoder from: {image_ckpt_path}")

In [ ]:
# Visualise image reconstruction
image_model.eval()
with torch.no_grad():
    batch_imgs = next(iter(image_val_loader))
    recon_imgs = image_model(batch_imgs.to(image_model.device)).cpu()

fig, axes = plt.subplots(2, min(4, len(batch_imgs)), figsize=(12, 6))
for i in range(min(4, len(batch_imgs))):
    axes[0, i].imshow(batch_imgs[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(recon_imgs[i].permute(1, 2, 0).numpy())
    axes[1, i].axis('off')
axes[0,0].set_title('Originals')
axes[1,0].set_title('Reconstructions')
plt.show()


Loss curve from the original training run's logs.

In [ ]:
metrics_path_img = ASTROCLIP_ROOT / 'logs' / 'image_autoencoder_notebook' / 'version_0' / 'metrics.csv'
if metrics_path_img.exists():
    df_img = pd.read_csv(metrics_path_img)
else:
    df_img = pd.DataFrame()

if not df_img.empty:
    plt.figure(figsize=(6, 4))
    if 'train_loss' in df_img.columns:
        plt.plot(df_img['epoch'], df_img['train_loss'], label='train_loss')
    if 'val_loss' in df_img.columns:
        plt.plot(df_img['epoch'], df_img['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Image AE loss curve (original training run)')
    plt.legend()
    plt.show()
else:
    print('No metrics found for plotting (expected if the checkpoint came from elsewhere).')

## 2. Spectrum Autoencoder (loaded from checkpoint)

In [ ]:
spectrum_ckpt_path = ASTROCLIP_ROOT / 'spectrum_autoencoder_notebook.ckpt'
spectrum_train_loader = build_spectrum_dataloader(ds['train'], batch_size=256, shuffle=True, num_workers=0)
spectrum_val_loader = build_spectrum_dataloader(ds['test'], batch_size=256, shuffle=False, num_workers=0)

if not spectrum_ckpt_path.exists():
    raise FileNotFoundError(
        f"{spectrum_ckpt_path} not found. Run astroclip_teaching_standalone.ipynb "
        "at least once first to produce this checkpoint."
    )

spectrum_model = SpectrumAutoencoder.load_from_checkpoint(spectrum_ckpt_path)
spectrum_model.eval()
print(f"Loaded pretrained spectrum autoencoder from: {spectrum_ckpt_path}")

In [ ]:
# Visualise spectrum reconstruction
spectrum_model.eval()
with torch.no_grad():
    batch_spec = next(iter(spectrum_val_loader))
    recon_spec = spectrum_model(batch_spec.to(spectrum_model.device)).cpu()

plt.figure(figsize=(10, 4))
plt.plot(batch_spec[0].numpy(), label='Original')
plt.plot(recon_spec[0].numpy(), label='Reconstruction')
plt.title('Spectrum reconstruction example')
plt.xlabel('Pixel')
plt.ylabel('Flux')
plt.legend()
plt.show()


In [ ]:
metrics_path_spec = ASTROCLIP_ROOT / 'logs' / 'spectrum_autoencoder_notebook' / 'version_0' / 'metrics.csv'
if metrics_path_spec.exists():
    df_spec = pd.read_csv(metrics_path_spec)
else:
    df_spec = pd.DataFrame()

if not df_spec.empty:
    plt.figure(figsize=(6, 4))
    if 'train_loss' in df_spec.columns:
        plt.plot(df_spec['epoch'], df_spec['train_loss'], label='train_loss')
    if 'val_loss' in df_spec.columns:
        plt.plot(df_spec['epoch'], df_spec['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Spectrum AE loss curve (original training run)')
    plt.legend()
    plt.show()
else:
    print('No metrics found for plotting (expected if the checkpoint came from elsewhere).')

## 3. CLIP Alignment (loaded from checkpoint)

In [ ]:
clip_ckpt_path = ASTROCLIP_ROOT / 'clip_alignment_notebook.ckpt'
clip_train_loader = build_multimodal_dataloader(ds['train'], batch_size=256, shuffle=True, num_workers=0)
clip_val_loader = build_multimodal_dataloader(ds['test'], batch_size=256, shuffle=False, num_workers=0)

if not clip_ckpt_path.exists():
    raise FileNotFoundError(
        f"{clip_ckpt_path} not found. Run astroclip_teaching_standalone.ipynb "
        "at least once first to produce this checkpoint."
    )

image_encoder = ImageAutoencoder.load_from_checkpoint(image_ckpt_path)
spectrum_encoder = SpectrumAutoencoder.load_from_checkpoint(spectrum_ckpt_path)

clip_model = SmallCLIPModel.load_from_checkpoint(
    clip_ckpt_path,
    image_encoder=image_encoder,
    spectrum_encoder=spectrum_encoder,
)
clip_model.eval()
print(f"Loaded pretrained CLIP alignment model from: {clip_ckpt_path}")

In [ ]:
metrics_path_clip = ASTROCLIP_ROOT / 'logs' / 'clip_alignment_notebook' / 'version_0' / 'metrics.csv'
if metrics_path_clip.exists():
    df_clip = pd.read_csv(metrics_path_clip)
else:
    df_clip = pd.DataFrame()

if not df_clip.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    if 'train_loss' in df_clip.columns:
        ax.plot(df_clip['epoch'], df_clip['train_loss'], label='train_loss')
    if 'val_loss' in df_clip.columns:
        ax.plot(df_clip['epoch'], df_clip['val_loss'], label='val_loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('CLIP alignment loss curve (original training run)')
    ax.legend(loc='upper left')
    if 'logit_scale' in df_clip.columns:
        ax2 = ax.twinx()
        ax2.plot(df_clip['epoch'], df_clip['logit_scale'], color='tab:green', linestyle='--', label='logit_scale')
        ax2.set_ylabel('Logit scale')
        ax2.legend(loc='lower right')
    plt.show()
else:
    print('No metrics found for plotting (expected if the checkpoint came from elsewhere).')

## 4. Embed Validation Split

Generate embeddings for the test split using the loaded (already-trained)
CLIP model — this is just a forward pass, no training.

In [ ]:
clip_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clip_model.to(device)

val_loader = build_multimodal_dataloader(ds['test'], batch_size=256, shuffle=False, num_workers=0)

image_embeddings = []
spectrum_embeddings = []
redshifts = []
targetids = []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        spectra = batch['spectrum'].to(device)
        img_emb, sp_emb = clip_model(images, spectra)
        image_embeddings.append(img_emb.cpu().numpy())
        spectrum_embeddings.append(sp_emb.cpu().numpy())
        if 'redshift' in batch:
            redshifts.append(batch['redshift'].numpy())
        if 'targetid' in batch:
            targetids.append(batch['targetid'].numpy())

image_embeddings = np.concatenate(image_embeddings, axis=0)
spectrum_embeddings = np.concatenate(spectrum_embeddings, axis=0)
redshift = np.concatenate(redshifts, axis=0) if redshifts else None
targetid = np.concatenate(targetids, axis=0) if targetids else None

print(image_embeddings.shape, spectrum_embeddings.shape)


### UMAP Projection

Visualise the joint embedding space.

In [ ]:
reducer = umap.UMAP(random_state=42)
joint = reducer.fit_transform(np.concatenate([image_embeddings, spectrum_embeddings], axis=0))
labels = np.concatenate([np.zeros(len(image_embeddings)), np.ones(len(spectrum_embeddings))])

plt.figure(figsize=(6, 5))
plt.scatter(joint[labels == 0, 0], joint[labels == 0, 1], s=5, alpha=0.5, label='Images')
plt.scatter(joint[labels == 1, 0], joint[labels == 1, 1], s=5, alpha=0.5, label='Spectra')
plt.legend()
plt.title('UMAP of joint embedding space')
plt.show()

## 5. Downstream Redshift Regression

Train a tiny MLP on the (pretrained) embeddings. This part is cheap
regardless — small model, small data — so it still trains live in ~seconds,
it's only the three encoders above that are loaded pretrained.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

features = np.concatenate([image_embeddings, spectrum_embeddings], axis=1)
labels = redshift if redshift is not None else np.zeros(len(features))

split = int(0.8 * len(features))
train_X, val_X = features[:split], features[split:]
train_y, val_y = labels[:split], labels[split:]

train_ds = TensorDataset(torch.from_numpy(train_X).float(), torch.from_numpy(train_y).float())
val_ds = TensorDataset(torch.from_numpy(val_X).float(), torch.from_numpy(val_y).float())
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLP(torch.nn.Module):
    def __init__(self, input_dim, hidden=256):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden // 2),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = MLP(features.shape[1]).to(dev)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

train_losses, val_losses = [], []
for epoch in range(50):
    model.train()
    running = 0.0
    for batch_x, batch_y in train_dl:
        batch_x, batch_y = batch_x.to(dev), batch_y.to(dev)
        optimizer.zero_grad(set_to_none=True)
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running += loss.item() * batch_x.size(0)
    train_loss = running / len(train_dl.dataset)

    model.eval()
    running = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_dl:
            batch_x, batch_y = batch_x.to(dev), batch_y.to(dev)
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            running += loss.item() * batch_x.size(0)
    val_loss = running / len(val_dl.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Redshift regression losses')
plt.legend()
plt.show()


In [ ]:
# Predicted vs. true redshift
model.eval()
preds = []
with torch.no_grad():
    for batch_x, _ in val_dl:
        preds.extend(model(batch_x.to(dev)).cpu().numpy())
preds = np.array(preds)
true = val_y[: len(preds)]

plt.figure(figsize=(5, 5))
plt.scatter(true, preds, s=8, alpha=0.6)
lims = [true.min(), true.max()]
plt.plot(lims, lims, '--', color='gray')
plt.xlabel('True redshift')
plt.ylabel('Predicted redshift')
plt.title('Redshift predictions')
plt.show()


## 6. Recap

This quicklook notebook loaded pretrained image/spectrum/CLIP checkpoints
instead of training them live, then reproduced the same reconstructions,
joint embedding space, and downstream redshift regression as the full
`astroclip_teaching_standalone.ipynb`. Use it as a backup if live training
runs out of time in class, or to skip straight to discussing results.

For deeper dives (same notes as the full notebook):
- Increase subset size / epochs for better performance, then re-run the full
  notebook to refresh these checkpoints.
- Replace autoencoders with the full AstroDINO/SpecFormer models.
- Pull in Galaxy Zoo morphology labels once RA/Dec bridges are available.